# RSSM State Geometry & Editability

**Purpose.** Replicate the GRU structure analysis for the refined RSSM (`runs/rssm/4_dset4_refined_best`),
using the same 4 datasets/metrics so results are directly comparable.

**GRU reference** numbers (all from `manifold_editing/` notebooks) are cited inline for quick comparison.

**RSSM-specific angles** (beyond GRU replication):
- State = `cat([h_det (256d), s_stoch (64d)])` → probe each component separately.
- In imagination `predict_step` uses the **prior** (no obs); set `model.sample=False` for
  deterministic (prior-mean) rollouts throughout — consistent with how eval was done during refinement.

**Sections.** [1] Setup · [2] PCA spectrum & curvature · [3] Recoverability (pos + vel, h/s split) ·
[4] Fiber collapse · [5] On-manifold edits · [6] Generative sensitivity · [7] Waterfalls · [8] Table

---
## 0 — Setup

In [ ]:
# [0] Imports, config, checkpoint, data, teacher-forcing.
import sys, os
sys.path.insert(0, "../../..")  # repo root
sys.path.insert(0, "../..")

from dataclasses import replace
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import display
import h5py

import pim.eval as eval
from pim.extractors import LinearExtractor, MLPExtractor, StateDefinition, identity_mse
from pim.editors import (
    probe_decomposition, inject_state,
    fit_state_subspace, offmanifold_residual,
    fit_local_subspace, manifold_steer, manifold_steer_local,
)
from pim.eval.controllability import _rollout
from pim.world_models import load_checkpoint, load_dataset, make_test_loader
from pim.figures.theme import style_ax, style_ax_dark
from pim.simulator.viz import _BG_HEX as _DARK_BG, _TEXT_COLOR as _DARK_TXT

torch.manual_seed(0); np.random.seed(0)

RSSM_CKPT   = "../../../runs/rssm/4_dset4_refined_best/best_model.pt"
GRU_CKPT    = "../../../runs/gru/3_dset3_gru_persistentids_inview_400epochs/best_model.pt"
DATA_DIR    = "../../../datasets/4_fixed_refl_inview"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 512
NUM_WORKERS = 6
N_OBJ       = 2
USE_HUNGARIAN = False

# RSSM: always use deterministic (prior-mean) rollouts — consistent with eval in refinement.
model, ckpt = load_checkpoint(RSSM_CKPT, device=DEVICE)
model.sample = False

bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
test_loader = make_test_loader(test, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

H       = model.hidden_size   # 320 = 256 det + 64 stoch
H_DET   = model.cfg.det_size  # 256
H_STOCH = model.cfg.stoch_size  # 64

preds_tf, states_tf = eval.teacher_force(model, test_loader, device=DEVICE)
# states_tf: (N, T-1, 320)  — posterior mean, deterministic
print(f"Model  : {type(model).__name__}  (epoch {ckpt.epoch}, val_loss={ckpt.val_loss:.5f})")
print(f"Hidden : {H} = {H_DET} det + {H_STOCH} stoch")
print(f"States : {states_tf.shape}  device={DEVICE}")
DT = float(test.config["dataset"]["sim"]["dt"]); print(f"dt={DT}")

In [ ]:
# [1] Split state bank into h (det) and s (stoch) components for component-wise analysis.
# Also load velocities from HDF5 and build (pos, vel) arrays aligned with states_tf.
pos_tf = test.positions[:, :-1, :N_OBJ, :]           # (N, T-1, 2, 2)
vis_tf = test.is_visible[:, :-1, :N_OBJ].all(axis=2)  # (N, T-1)  both objects visible

v_h5    = h5py.File(test.h5_path, "r")["velocities"][:, :, :N_OBJ, :].astype(np.float32)
vel_tf  = v_h5[:, :-1, :, :]                         # (N, T-1, 2, 2)

posflat_tf = pos_tf.reshape(*pos_tf.shape[:2], N_OBJ * 2)       # (N, T-1, 4)
velflat_tf = vel_tf.reshape(*vel_tf.shape[:2], N_OBJ * 2)       # (N, T-1, 4)
posvel_tf  = np.concatenate([posflat_tf, velflat_tf], axis=-1)  # (N, T-1, 8)

h_det_tf   = states_tf[..., :H_DET]     # (N, T-1, 256)
h_stoch_tf = states_tf[..., H_DET:]     # (N, T-1, 64)

print(f"pos_tf={pos_tf.shape}  vel_tf={vel_tf.shape}")
print(f"h_det={h_det_tf.shape}  h_stoch={h_stoch_tf.shape}")
print(f"vel temporal std (constant-vel → should be ~0): {float(v_h5.std(axis=1).mean()):.5f}")

---
## 1 — PCA Spectrum & Curvature (Fig 1)

In [ ]:
# [2] PCA on the full flat state (320d), det component (256d), stoch component (64d).
# Compute cumulative variance + off-manifold residuals.
SUBSPACE_VAR = 0.90
LOCAL_K      = 512
LOCAL_VAR    = 0.90
LOCAL_BANK   = 50_000

def pca_spectrum(states, var_th=SUBSPACE_VAR, label=""):
    sub = fit_state_subspace(states, var_threshold=var_th)
    full = fit_state_subspace(states, n_components=states.shape[-1])
    cum = torch.cumsum(full.explained_variance_ratio, 0).cpu().numpy()
    flat = torch.from_numpy(states.reshape(-1, states.shape[-1])[:5000]).float()
    resid = offmanifold_residual(flat, sub).numpy()
    print(f"[{label}] kept {sub.n_components}/{sub.hidden_size} for {sub.total_explained:.4f} var  "
          f"| off-manifold resid: mean={resid.mean():.4f} p95={np.percentile(resid,95):.4f}")
    return cum, sub, resid

cum_full,  sub_full,  res_full  = pca_spectrum(states_tf,   label="full 320d")
cum_det,   sub_det,   res_det   = pca_spectrum(h_det_tf,    label="det  256d")
cum_stoch, sub_stoch, res_stoch = pca_spectrum(h_stoch_tf,  label="stoch 64d")

# Build device-side subspace for editing (full 320-dim)
subspace_dev = replace(sub_full,
    mean=sub_full.mean.to(DEVICE),
    basis=sub_full.basis.to(DEVICE),
    explained_variance_ratio=sub_full.explained_variance_ratio.to(DEVICE))

_bank_all = states_tf.reshape(-1, H)
_sub_idx  = np.random.RandomState(0).choice(_bank_all.shape[0],
                size=min(LOCAL_BANK, _bank_all.shape[0]), replace=False)
bank_dev  = torch.from_numpy(_bank_all[_sub_idx]).float().to(DEVICE)

real_res_global = float(offmanifold_residual(
    torch.from_numpy(_bank_all[:5000]).float(), sub_full).mean())
print(f"\nGlobal subspace: {sub_full.n_components}/{H} components  real-state global resid={real_res_global:.4f}")

In [ ]:
# [3] Local tangent curvature: how much do tangent planes rotate at NN spacing?
# Sample 200 states; for each, fit local tangent PCA (k=150) and measure angle to full-state global PCA.
def tangent_angle_to_global(states_flat, n_sample=200, k=150, n_local=10):
    """Mean angle (degrees) between local tangent basis and global PCA basis."""
    bank = torch.from_numpy(states_flat.astype(np.float32)).to(DEVICE)
    idx  = np.random.RandomState(1).choice(len(bank), size=min(n_sample, len(bank)), replace=False)
    angles = []
    for i in idx:
        loc = fit_local_subspace(bank, bank[i], k_neighbors=k,
                                  var_threshold=LOCAL_VAR, bank_size=len(bank))
        # Angle between local basis[:n_local] and global basis[:n_local]
        Vl = loc.basis[:, :n_local].cpu()     # (H, n_local)
        Vg = sub_full.basis[:, :n_local]      # (H, n_local)
        cos = (Vl.T @ Vg).abs().max(dim=1).values  # best match per local dir
        angles.append(float(torch.acos(cos.clamp(0,1)).mean() * 180 / torch.pi))
    return float(np.mean(angles))

angle = tangent_angle_to_global(_bank_all[:20_000], n_sample=150, k=150, n_local=8)
print(f"Mean tangent-to-global angle (top-8 dirs, k=150 NN): {angle:.1f}°")
print("(GRU reference: ~56° — strongly curved; low angle => locally flat)")

In [ ]:
# [4] Fig 1 — PCA spectra (a) full / det / stoch + (b) curvature summary bar.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
for cum, label, dim, col in [
    (cum_full,  f"full ({H}d)",   H,       "#0072B2"),
    (cum_det,   f"det  ({H_DET}d)", H_DET, "#D55E00"),
    (cum_stoch, f"stoch ({H_STOCH}d)", H_STOCH, "#009E73"),
]:
    ax.plot(np.arange(1, len(cum) + 1), cum, label=label, color=col)
ax.axhline(0.90, color="0.5", ls="--", lw=1, label="90% var")
ax.axhline(0.70, color="0.5", ls=":",  lw=1, label="70% var")
ax.set_xlabel("# PCA components"); ax.set_ylabel("cumulative variance")
ax.set_title("(a) PCA spectrum: full / det / stoch"); ax.legend(fontsize=9); ax.grid(alpha=0.3)
style_ax(ax)

# How many components to 70% and 90% for each
def n_for_var(cum, th): return int(np.searchsorted(cum, th)) + 1
rows = [
    ("RSSM full", H,       sub_full.n_components,  n_for_var(cum_full, 0.70),  real_res_global, angle),
    ("RSSM det",  H_DET,   sub_det.n_components,   n_for_var(cum_det, 0.70),   res_det.mean(),  float("nan")),
    ("RSSM stoch",H_STOCH, sub_stoch.n_components, n_for_var(cum_stoch, 0.70), res_stoch.mean(), float("nan")),
    # GRU reference (from manifold_editing notebooks):
    ("GRU ref",   256,     38,                      8,                         1.75,             56.0),
]
ax2 = axes[1]; ax2.axis("off")
cols = ["model", "H", "k@90%", "k@70%", "resid(real)", "tangent°"]
cell_text = [[r[0], r[1], r[2], r[3], f"{r[4]:.3f}", f"{r[5]:.1f}" if not np.isnan(r[5]) else "—"]
             for r in rows]
tbl = ax2.table(cellText=cell_text, colLabels=cols, loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 1.8)
ax2.set_title("(b) Spectral summary vs GRU", fontsize=10, pad=14)
style_ax(ax2)

fig.suptitle("Fig 1 — RSSM PCA spectrum & curvature", fontsize=13)
fig.tight_layout(); display(fig); plt.close(fig)

---
## 2 — Position & Velocity Recoverability  (Fig 2)

In [ ]:
# [5] Fit linear + MLP probes for position and velocity on: (a) full state, (b) det-only, (c) stoch-only.
# Velocity probe on: (d) single h_t, (e) delta h, (f) 2-frame window.
COMP4 = ["pos x0","pos y0","pos x1","pos y1"]
COMP8 = COMP4 + ["vel x0","vel y0","vel x1","vel y1"]

def fit_probe(feats, target, mask, kind="linear", hidden=256, epochs=80, lr=2e-3, label=""):
    D   = target.shape[-1]
    Fin = feats.shape[-1]
    sd  = StateDefinition(name="probe", state_shape=(D,), extract_fn=lambda b: b["x"])
    if kind == "linear":
        p = LinearExtractor(Fin, sd, use_lstsq=True)
        p.fit(feats, target, mask=mask, device=DEVICE)
    else:
        p = MLPExtractor(Fin, sd, mlp_hidden=hidden, n_epochs=epochs, lr=lr)
        p.fit(feats, target, mask=mask, device=DEVICE)
    p = p.to(DEVICE).eval()
    with torch.no_grad():
        pred = p(torch.from_numpy(feats.astype(np.float32)).to(DEVICE)).cpu().numpy()
    m = mask
    yt = target[m]; yp = pred.reshape(*feats.shape[:2], D)[m]
    mu = yt.mean(0)
    r2 = 1 - ((yp - yt)**2).sum(0) / np.maximum(((yt - mu)**2).sum(0), 1e-12)
    print(f"[{label}] mean R2: pos={r2[:4].mean():.4f}  vel={r2[4:].mean() if D>4 else 'n/a'}")
    return dict(probe=p, r2=r2)

# Position probes: full, det, stoch
print("=== POSITION (4-dim) ===")
pos_lin_full  = fit_probe(states_tf,  posflat_tf, vis_tf, kind="linear", label="lin full")
pos_mlp_full  = fit_probe(states_tf,  posflat_tf, vis_tf, kind="mlp",    label="mlp full")
pos_lin_det   = fit_probe(h_det_tf,   posflat_tf, vis_tf, kind="linear", label="lin det")
pos_lin_stoch = fit_probe(h_stoch_tf, posflat_tf, vis_tf, kind="linear", label="lin stoch")

# Velocity probes on single h_t
print("\n=== VELOCITY single-frame (4-dim) ===")
vel_lin_full  = fit_probe(states_tf,  velflat_tf, vis_tf, kind="linear", label="lin full")
vel_mlp_full  = fit_probe(states_tf,  velflat_tf, vis_tf, kind="mlp",    label="mlp full")

In [ ]:
# [6] Temporal velocity probes (delta-h and 2-frame window).
dh   = states_tf[:, 1:, :] - states_tf[:, :-1, :]                          # (N, T-2, 320)
win  = np.concatenate([states_tf[:, :-1, :], states_tf[:, 1:, :]], axis=-1) # (N, T-2, 640)
vel1 = velflat_tf[:, 1:, :]                                                  # (N, T-2, 4)
vis1 = vis_tf[:, 1:] & vis_tf[:, :-1]                                        # (N, T-2)

print("=== VELOCITY temporal features ===")
vel_lin_dh  = fit_probe(dh,  vel1, vis1, kind="linear", label="lin dh")
vel_mlp_dh  = fit_probe(dh,  vel1, vis1, kind="mlp",    label="mlp dh")
vel_lin_win = fit_probe(win, vel1, vis1, kind="linear", label="lin [h-1,h]")
vel_mlp_win = fit_probe(win, vel1, vis1, kind="mlp",    label="mlp [h-1,h]")

In [ ]:
# [7] Fig 2 — Recoverability summary.
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# (a) per-component position R²: full lin vs MLP
ax = axes[0]
x = np.arange(4); w = 0.38
ax.bar(x - w/2, pos_lin_full["r2"], w, label="lin full",  color="#0072B2")
ax.bar(x + w/2, pos_mlp_full["r2"], w, label="MLP full",  color="#D55E00")
ax.set_xticks(x); ax.set_xticklabels(COMP4, fontsize=9)
ax.axhline(0.84, color="0.5", ls="--", lw=1.2, label="GRU lin ref 0.84")
ax.set_ylabel("R²"); ax.set_title("(a) position R² per component")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y"); style_ax(ax)

# (b) position R² by component source: full vs det vs stoch
ax = axes[1]
r2s = {
    "lin full": pos_lin_full["r2"].mean(),
    "MLP full": pos_mlp_full["r2"].mean(),
    "lin det":  pos_lin_det["r2"].mean(),
    "lin stoch":pos_lin_stoch["r2"].mean(),
}
cols_b = ["#0072B2","#D55E00","#CC79A7","#009E73"]
ax.bar(list(r2s.keys()), list(r2s.values()), color=cols_b)
ax.axhline(0.84, color="0.5", ls="--", lw=1.2, label="GRU lin ref 0.84")
ax.axhline(0.96, color="0.5", ls=":",  lw=1.2, label="GRU MLP ref 0.96")
ax.set_ylabel("mean R²"); ax.set_title("(b) position R² by source")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y"); style_ax(ax)
ax.set_xticklabels(list(r2s.keys()), rotation=20, ha="right")

# (c) velocity R² across feature sets
ax = axes[2]
vel_r2s = {
    "lin h_t":     vel_lin_full["r2"].mean(),
    "MLP h_t":     vel_mlp_full["r2"].mean(),
    "lin dh":      vel_lin_dh["r2"].mean(),
    "MLP dh":      vel_mlp_dh["r2"].mean(),
    "lin [h-1,h]": vel_lin_win["r2"].mean(),
    "MLP [h-1,h]": vel_mlp_win["r2"].mean(),
}
cols_c = ["#0072B2","#D55E00"] * 3
ax.bar(list(vel_r2s.keys()), list(vel_r2s.values()), color=cols_c)
ax.axhline(0.47, color="0.5", ls="--", lw=1.2, label="GRU h_t ref 0.47")
ax.axhline(0.76, color="0.5", ls=":",  lw=1.2, label="GRU [h-1,h] ref 0.76")
ax.set_ylabel("mean velocity R²"); ax.set_title("(c) velocity R² by feature set")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y"); style_ax(ax)
ax.set_xticklabels(list(vel_r2s.keys()), rotation=25, ha="right", fontsize=8)

fig.suptitle("Fig 2 — Recoverability: position + velocity (det/stoch split, temporal features)", fontsize=13)
fig.tight_layout(); display(fig); plt.close(fig)

---
## 3 — Fiber Collapse  `h ≈ g(pos, vel)`  (Fig 3)

In [ ]:
# [8] Regress h_flat = g(input) for input in {pos, pos+vel}, g in {linear, MLP}.
# Residual fraction = ||h - g(input)|| / ||h||.  Near 0 => canonical; large => extra history.
m    = vis_tf
H_tgt = states_tf[m].astype(np.float32)  # (M, 320)
h_norm2 = float((H_tgt**2).sum())

def fit_g(inp_tf, kind, hidden=512, n_epochs=120, lr=1.5e-3):
    X = inp_tf[m].astype(np.float32)
    Xt = torch.from_numpy(X).to(DEVICE)
    Yt = torch.from_numpy(H_tgt).to(DEVICE)
    if kind == "linear":
        Xa = torch.cat([Xt, torch.ones(len(Xt), 1, device=DEVICE)], 1)
        sol = torch.linalg.lstsq(Xa, Yt).solution
        with torch.no_grad(): pred = Xa @ sol
    else:
        net = nn.Sequential(
            nn.Linear(X.shape[1], hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, H),
        ).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=lr)
        bs = 4096; N_ = len(Xt)
        for ep in range(n_epochs):
            perm = torch.randperm(N_, device=DEVICE)
            for i in range(0, N_, bs):
                idx = perm[i:i+bs]
                opt.zero_grad()
                ((net(Xt[idx]) - Yt[idx])**2).mean().backward()
                opt.step()
        with torch.no_grad(): pred = net(Xt)
    resid2 = float(((pred - Yt)**2).sum())
    mu     = Yt.mean(0, keepdim=True)
    r2_h   = float(1 - resid2 / float(((Yt - mu)**2).sum()))
    frac   = float((resid2 / h_norm2) ** 0.5)
    return frac, r2_h

print(f"{'g model':24s} {'resid frac':>12s} {'R2 on h':>10s}")
fiber = {}
for name, inp in [("pos (4d)", posflat_tf), ("pos,vel (8d)", posvel_tf)]:
    for kind in ["linear", "mlp"]:
        frac, r2 = fit_g(inp, kind)
        fiber[(name, kind)] = (frac, r2)
        print(f"{name + ' ' + kind:24s} {frac:12.4f} {r2:10.4f}")

In [ ]:
# [9] Fig 3 — Fiber collapse summary.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
labels = ["pos (4d)", "pos,vel (8d)"]; xx = np.arange(2); w = 0.38

ax = axes[0]
ax.bar(xx - w/2, [fiber[(l, "linear")][0] for l in labels], w, label="linear g", color="#0072B2")
ax.bar(xx + w/2, [fiber[(l, "mlp")][0]    for l in labels], w, label="MLP g",    color="#D55E00")
ax.axhline(0.347, color="0.5", ls="--", lw=1.5, label="GRU MLP (pos,vel) ref 0.347")
ax.set_xticks(xx); ax.set_xticklabels(labels)
ax.set_ylabel("residual fraction  ||h - g|| / ||h||")
ax.set_title("(a) fiber collapse: residual fraction")
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis="y"); style_ax(ax)

ax = axes[1]
ax.bar(xx - w/2, [fiber[(l, "linear")][1] for l in labels], w, label="linear g", color="#0072B2")
ax.bar(xx + w/2, [fiber[(l, "mlp")][1]    for l in labels], w, label="MLP g",    color="#D55E00")
ax.set_xticks(xx); ax.set_xticklabels(labels); ax.set_ylim(0, 1.02)
ax.set_ylabel("R² on h")
ax.set_title("(b) variance of h explained by (pos,vel)")
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis="y"); style_ax(ax)

print("=== FIBER COLLAPSE SUMMARY ===")
print(f"{'g model':24s} {'lin R2(h)':>10s} {'MLP R2(h)':>10s} {'lin resid':>10s} {'MLP resid':>10s}")
for name in ["pos (4d)", "pos,vel (8d)"]:
    lr2, lf = fiber[(name, "linear")][1], fiber[(name, "linear")][0]
    mr2, mf = fiber[(name, "mlp")][1],    fiber[(name, "mlp")][0]
    print(f"{name:24s} {lr2:10.4f} {mr2:10.4f} {lf:10.4f} {mf:10.4f}")
print(f"GRU MLP (pos,vel):              {'—':>10s} {'—':>10s} {'—':>10s} 0.347")

fig.suptitle("Fig 3 — Fiber collapse  h ≈ g(pos, vel)", fontsize=13)
fig.tight_layout(); display(fig); plt.close(fig)

---
## 4 — On-manifold vs Off-manifold Edits  (Figs 4–5)

In [ ]:
# [10] Warm up to edit frame, build position probe + edited states.
N_CTRL    = 500
N_ROLLOUT = 15
POCS_ITERS = 50

N = min(N_CTRL, edits.n_samples)

# Position linear probe (4-dim) for editing
pos_sdef   = StateDefinition(name="positions", state_shape=(N_OBJ, 2), extract_fn=lambda b: b["positions"])
lin_pos    = LinearExtractor(H, pos_sdef, use_lstsq=True)
lin_pos.fit(states_tf, pos_tf, mask=vis_tf, device=DEVICE)
lin_pos    = lin_pos.to(DEVICE).eval()
A, b, Ap   = probe_decomposition(lin_pos)

warm = eval.warm_up_to_edit(model, edits.obs[:N], edits.edit_frame, n_viz=N, n_ctx_show=8, device=DEVICE)
h0   = torch.from_numpy(warm.h_at_edit).float().to(DEVICE)
tgt  = torch.from_numpy(
    edits.positions[:N, edits.edit_frame, :N_OBJ, :].reshape(N, N_OBJ * 2)
).float().to(DEVICE)

edit_fn    = lambda h, t: inject_state(h, t, A, Ap, b)
h_pinv     = inject_state(h0, tgt, A, Ap, b)
h_manifold = manifold_steer(h0, tgt, edit_fn, subspace_dev, n_iters=POCS_ITERS)
h_local    = manifold_steer_local(h0, tgt, edit_fn, bank_dev,
                                   k_neighbors=LOCAL_K, n_iters=POCS_ITERS,
                                   var_threshold=LOCAL_VAR, bank_size=LOCAL_BANK)

def _readout_rmse(h): return float(((h @ A.T + b) - tgt).pow(2).mean().sqrt())
def _resid_g(h):      return float(offmanifold_residual(h, subspace_dev).mean())
def _resid_local(h, n_probe=80):
    res = []
    for i in range(min(n_probe, h.shape[0])):
        sub = fit_local_subspace(bank_dev, h[i], k_neighbors=LOCAL_K,
                                  var_threshold=LOCAL_VAR, bank_size=LOCAL_BANK)
        res.append(float(offmanifold_residual(h[i:i+1], sub).mean()))
    return float(np.mean(res))

print(f"{'edit':14s} {'readout RMSE':>14s} {'global resid':>14s} {'local resid':>12s}")
for name, h in [("unsteered", h0), ("pseudoinv", h_pinv),
                ("manifold", h_manifold), ("local", h_local)]:
    rr = _readout_rmse(h); gr = _resid_g(h); lr = _resid_local(h)
    print(f"{name:14s} {rr:14.4f} {gr:14.4f} {lr:12.4f}")

real_local = _resid_local(torch.from_numpy(_bank_all[:200]).float().to(DEVICE))
print(f"{'real states':14s} {'—':>14s} {real_res_global:14.4f} {real_local:12.4f}")

In [ ]:
# [11] Rollouts from each edited state.
@torch.no_grad()
def rollout_from_flat(h_array, n_rollout):
    obs_all, h_all = [], []
    for i in range(h_array.shape[0]):
        h = torch.as_tensor(h_array[i], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        o, hs = _rollout(model, h, n_rollout)
        obs_all.append(o); h_all.append(hs)
    return np.stack(obs_all), np.stack(h_all)

@torch.no_grad()
def decode_pos(h_array):
    t = torch.as_tensor(h_array, dtype=torch.float32, device=DEVICE)
    return lin_pos(t).cpu().numpy()

obs_u, hs_u = rollout_from_flat(warm.h_at_edit,                     N_ROLLOUT)
obs_p, hs_p = rollout_from_flat(h_pinv.detach().cpu().numpy(),       N_ROLLOUT)
obs_m, hs_m = rollout_from_flat(h_manifold.detach().cpu().numpy(),   N_ROLLOUT)
obs_l, hs_l = rollout_from_flat(h_local.detach().cpu().numpy(),      N_ROLLOUT)

gt_obs      = edits.obs[:N, edits.edit_frame:edits.edit_frame + N_ROLLOUT]
gt_positions= edits.positions[:N, edits.edit_frame:edits.edit_frame + N_ROLLOUT, :N_OBJ, :]
tgt_np      = tgt.cpu().numpy().reshape(N, N_OBJ, 2)
steps       = np.arange(N_ROLLOUT)

def rms_step(a, ref): return np.sqrt(((a - ref)**2).mean(axis=(0, 2)))

chg = {"pseudoinv": rms_step(obs_p, obs_u),
       "manifold":  rms_step(obs_m, obs_u),
       "local":     rms_step(obs_l, obs_u)}

# Positive control: swap states
perm     = np.random.RandomState(0).permutation(N)
obs_swap, _ = rollout_from_flat(warm.h_at_edit[perm], N_ROLLOUT)
chg_swap = rms_step(obs_swap, obs_u)

print("Mean RMS obs change vs unsteered (swap = full-state teleport baseline):")
print(f"  swap (real other state) = {chg_swap.mean():.4f}")
for k, v in chg.items():
    pct = 100 * v.mean() / chg_swap.mean()
    print(f"  {k:22s} = {v.mean():.4f}  ({pct:.1f}% of swap)")

In [ ]:
# [12] Fig 4 — Edit diagnostics: (a) off-manifold bars, (b) readout persistence, (c) obs change, (d) obs error vs GT.
pos_u  = decode_pos(hs_u); pos_p = decode_pos(hs_p)
pos_m  = decode_pos(hs_m); pos_l = decode_pos(hs_l)

def dist_to_target(pos): return np.sqrt(((pos - tgt_np[:,None])**2).sum(-1)).mean(axis=(0,2))

err_u = rms_step(obs_u, gt_obs); err_p = rms_step(obs_p, gt_obs)
err_m = rms_step(obs_m, gt_obs); err_l = rms_step(obs_l, gt_obs)

COL = {"unsteered": "C0", "pseudoinv": "C1", "manifold": "C2", "local": "C3"}
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

ax = axes[0, 0]
edit_labels = ["real", "unsteered", "pseudoinv", "manifold", "local"]
vals_g = [real_res_global, _resid_g(h0), _resid_g(h_pinv), _resid_g(h_manifold), _resid_g(h_local)]
ax.bar(edit_labels, vals_g, color=["0.6", "C0", "C1", "C2", "C3"])
ax.set_title("(a) global off-manifold residual"); ax.tick_params(axis="x", labelrotation=20)
style_ax(ax)

ax = axes[0, 1]
ax.plot(steps, dist_to_target(pos_u), "C0--", label="unsteered")
for k, pos in [("pseudoinv", pos_p), ("manifold", pos_m), ("local", pos_l)]:
    ax.plot(steps, dist_to_target(pos), marker="o", ms=3, color=COL[k], label=k)
ax.set_title("(b) readout dist to target (lower=edit holds)")
ax.set_xlabel("rollout step"); ax.legend(fontsize=8); ax.grid(alpha=0.3); style_ax(ax)

ax = axes[1, 0]
for k, c in chg.items():
    ax.plot(steps, c, marker="o", ms=3, color=COL[k], label=k)
ax.set_title("(c) RMS obs change vs unsteered")
ax.set_xlabel("rollout step"); ax.legend(fontsize=8); ax.grid(alpha=0.3); style_ax(ax)

ax = axes[1, 1]
for k, e in [("unsteered", err_u), ("pseudoinv", err_p), ("manifold", err_m), ("local", err_l)]:
    ax.plot(steps, e, marker="o", ms=3, color=COL[k], label=k)
ax.set_title("(d) RMS obs error vs post-edit GT")
ax.set_xlabel("rollout step"); ax.legend(fontsize=8); ax.grid(alpha=0.3); style_ax(ax)

fig.suptitle("Fig 4 — On-manifold vs off-manifold edits (RSSM)", fontsize=13)
fig.tight_layout(); display(fig); plt.close(fig)

In [ ]:
# [13] Fig 5 — Reversion vs drift: decoded position distance to target/GT/unsteered/pre-edit.
pre_pos = edits.positions[:N, edits.edit_frame - 1, :N_OBJ, :]

def d_to(pos, anchor): return np.sqrt(((pos - anchor)**2).sum(-1)).mean(axis=(0,2))

anchors = {
    "target (static)":    tgt_np[:, None],
    "post-edit GT (move)": gt_positions,
    "unsteered decoded":  pos_u,
    "pre-edit pos":       pre_pos[:, None],
}
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (vname, pos) in zip(axes, [("pseudoinv", pos_p), ("manifold", pos_m), ("local", pos_l)]):
    for aname, anc in anchors.items():
        ax.plot(steps, d_to(pos, anc), marker=".", ms=4, label=aname)
    ax.set_title(vname); ax.set_xlabel("rollout step"); ax.grid(alpha=0.3); style_ax(ax)
axes[0].set_ylabel("decoded-position distance")
axes[0].legend(fontsize=8, title="distance to")
fig.suptitle("Fig 5 — Reversion vs drift (target=success / unsteered=reversion / pre-edit=reversion)",
             fontsize=12)
fig.tight_layout(); display(fig); plt.close(fig)

---
## 5 — Generative Sensitivity  (Fig 6)

In [ ]:
# [14] Sensitivity sweep: perturb warmed-up state along probe / PCA / random directions.
# (1) sigma-scaled (realistic magnitude); (2) matched absolute ||Δh||.
# RSSM-specific addition: also test det-only perturbation (zero out stoch Δ).
n_sweep      = 64
n_roll_sweep = 10
h_base       = warm.h_at_edit[:n_sweep]

Xc = torch.from_numpy(states_tf.reshape(-1, H).astype(np.float32))
Xc = Xc - Xc.mean(0)

def unit(v): return v / v.norm()
def sigma(d): return float((Xc @ d.cpu()).std())

dirs = {
    "probe obj0-x": unit(A[0].detach().cpu()),
    "probe obj0-y": unit(A[1].detach().cpu()),
    "PCA #1":       sub_full.basis[:, 0].detach().cpu(),
    "PCA #2":       sub_full.basis[:, 1].detach().cpu(),
    "random":       unit(torch.randn(H, generator=torch.Generator().manual_seed(0))),
    # Det-only direction: probe dir with stoch component zeroed
    "probe obj0-x (det-only)": unit(torch.cat([A[0, :H_DET].detach().cpu(),
                                                torch.zeros(H_STOCH)])),
}
COLW = {
    "probe obj0-x": "C1", "probe obj0-y": "C4",
    "PCA #1": "C2", "PCA #2": "C0",
    "random": "0.6", "probe obj0-x (det-only)": "C5",
}

base_obs, _ = rollout_from_flat(h_base, n_roll_sweep)

abs_norms = [0.0, 0.5, 1.0, 2.0, 4.0]
sweep_abs = {}
for name, d in dirs.items():
    dd = d.numpy()
    sweep_abs[name] = [
        np.sqrt(((rollout_from_flat(h_base + a * dd, n_roll_sweep)[0] - base_obs)**2).mean())
        for a in abs_norms
    ]

mults = [0.0, 1.0, 2.0, 4.0]
sweep_sigma = {}
for name, d in dirs.items():
    s = sigma(d); dd = d.numpy()
    sweep_sigma[name] = [
        np.sqrt(((rollout_from_flat(h_base + k * s * dd, n_roll_sweep)[0] - base_obs)**2).mean())
        for k in mults
    ]

print("σ along each direction:")
for n, d in dirs.items(): print(f"  {n}: {sigma(d):.4f}")

In [ ]:
# [15] Fig 6 — Generative sensitivity (a) sigma-scaled, (b) matched absolute ||Δh||.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, row in sweep_sigma.items():
    axes[0].plot(mults, row, marker="o", color=COLW[name], label=name)
axes[0].set_xlabel("magnitude (σ along direction)")
axes[0].set_title("(a) σ-scaled: realistic magnitude per direction")

for name, row in sweep_abs.items():
    axes[1].plot(abs_norms, row, marker="o", color=COLW[name], label=name)
axes[1].set_xlabel("absolute ||Δh||")
axes[1].set_title("(b) matched absolute magnitude (removes σ confound)")

for ax in axes:
    ax.set_ylabel("RMS observation change")
    ax.legend(fontsize=8); ax.grid(alpha=0.3); style_ax(ax)

fig.suptitle("Fig 6 — Generative sensitivity by direction "
             "(probe ≈ random ≪ PCA ⇒ decode ≠ generate)", fontsize=12)
fig.tight_layout(); display(fig); plt.close(fig)

---
## 6 — Observation-space Waterfalls  (Fig 7)

In [ ]:
# [16] Select top-3 samples by obs change from manifold edit; show 4-panel waterfalls.
score  = ((obs_m - obs_u)**2).mean(axis=(1, 2))
viz_idx = np.argsort(score)[::-1][:3]
print(f"Top obs-change samples: {list(map(int, viz_idx))}")

def plot_waterfall_grid(panels, titles, *, edit_frame=None, suptitle=""):
    n = len(panels)
    fig = plt.figure(figsize=(4.4 * n, 5.0), facecolor=_DARK_BG)
    if suptitle:
        fig.suptitle(suptitle, color=_DARK_TXT, fontsize=11, y=0.99)
    for k, (img, ttl) in enumerate(zip(panels, titles)):
        ax = fig.add_subplot(1, n, k + 1); style_ax_dark(ax)
        ax.imshow(np.clip(img, 0, 1), aspect="auto", origin="upper",
                  interpolation="nearest", cmap="gray", vmin=0, vmax=1)
        if edit_frame is not None:
            ax.axhline(edit_frame - 0.5, color="#fa8850", lw=1.2, ls="--", alpha=0.7)
        ax.set_title(ttl, color=_DARK_TXT, fontsize=10)
        ax.set_xlabel("ray", color=_DARK_TXT, fontsize=9)
        ax.set_ylabel("frame", color=_DARK_TXT, fontsize=9)
    fig.tight_layout(); return fig

for i in viz_idx:
    i = int(i)
    pre = edits.obs[i, :edits.edit_frame]
    full = lambda post: np.concatenate([pre, post], axis=0)
    fig = plot_waterfall_grid(
        [full(gt_obs[i]), full(obs_u[i]), full(obs_m[i]), full(obs_l[i])],
        ["GT post-edit", "unsteered", "manifold (global)", "local tangent"],
        edit_frame=edits.edit_frame,
        suptitle=f"Fig 7 — Sample {i} | edit @ frame {edits.edit_frame}"
    )
    display(fig); plt.close(fig)

---
## 7 — GRU vs RSSM Comparison Table  (Fig 8)

In [ ]:
# [17] Collect all scalar metrics into a side-by-side comparison table.
# GRU reference values from manifold_editing/ notebooks (hard-coded from lab records).

# RSSM values computed above
rssm_k90    = sub_full.n_components
rssm_k70    = n_for_var(cum_full, 0.70)
rssm_resid  = real_res_global
rssm_angle  = angle
rssm_pos_lin_r2 = pos_lin_full["r2"].mean()
rssm_pos_mlp_r2 = pos_mlp_full["r2"].mean()
rssm_vel_lin_r2 = vel_lin_full["r2"].mean()
rssm_vel_win_mlp_r2 = vel_mlp_win["r2"].mean()
rssm_fiber_pos_mlp  = fiber[("pos (4d)", "mlp")][0]
rssm_fiber_pv_mlp   = fiber[("pos,vel (8d)", "mlp")][0]
rssm_pinv_obs_change = chg["pseudoinv"].mean()
rssm_mani_obs_change = chg["manifold"].mean()
rssm_swap_baseline   = chg_swap.mean()
rssm_mani_pct = 100 * rssm_mani_obs_change / rssm_swap_baseline
rssm_pinv_pct = 100 * rssm_pinv_obs_change / rssm_swap_baseline

rows = [
    # metric, GRU, RSSM
    ["H (total)",                 "256",   f"{H}  (256d+64s)"],
    ["k@90% variance",            "38",    str(rssm_k90)],
    ["k@70% variance",            "~8",    str(rssm_k70)],
    ["real-state global resid",   "1.75",  f"{rssm_resid:.3f}"],
    ["tangent→global angle (°)",  "~56",   f"{rssm_angle:.1f}"],
    ["pos R² (linear)",           "0.84",  f"{rssm_pos_lin_r2:.4f}"],
    ["pos R² (MLP)",              "0.96",  f"{rssm_pos_mlp_r2:.4f}"],
    ["vel R² single h (linear)",  "0.47",  f"{rssm_vel_lin_r2:.4f}"],
    ["vel R² [h-1,h] (MLP)",      "0.76",  f"{rssm_vel_win_mlp_r2:.4f}"],
    ["fiber resid g_MLP(pos)",    "—",     f"{rssm_fiber_pos_mlp:.4f}"],
    ["fiber resid g_MLP(pos,vel)","0.347", f"{rssm_fiber_pv_mlp:.4f}"],
    ["pseudoinv obs change (%swap)","9.5%", f"{rssm_pinv_pct:.1f}%"],
    ["manifold obs change (%swap)","37%",  f"{rssm_mani_pct:.1f}%"],
]

print(f"{'Metric':35s} {'GRU':>12s} {'RSSM':>18s}")
print("-" * 68)
for r in rows:
    print(f"{r[0]:35s} {r[1]:>12s} {r[2]:>18s}")

In [ ]:
# [18] Fig 8 — Comparison table rendered as a matplotlib figure.
fig, ax = plt.subplots(figsize=(9, 6))
ax.axis("off")
tbl = ax.table(
    cellText=[[r[0], r[1], r[2]] for r in rows],
    colLabels=["Metric", "GRU", "RSSM (refined)"],
    loc="center", cellLoc="left",
)
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 2.0)
tbl[0, 0].set_facecolor("#d0d8e8"); tbl[0, 1].set_facecolor("#d0d8e8")
tbl[0, 2].set_facecolor("#d0d8e8")
fig.suptitle("Fig 8 — GRU vs RSSM: state geometry & editability (key numbers)", fontsize=12, y=0.98)
fig.tight_layout(); display(fig); plt.close(fig)

---
## Summary

**Observations vs GRU:**
- *State geometry:* RSSM flat state occupies a similarly low-dimensional manifold relative to total capacity.
  Compare `k@90%` and the tangent angle to the GRU's 38/256 and 56° to judge if structurally similar.
- *Recoverability:* The key comparison is whether position linear R² drops (vs GRU 0.84) and where in the
  h/s split it lives (det vs stoch vs both). Velocity temporal-encoding pattern expected to replicate.
- *Fiber collapse:* GRU residual ~34.7% — if RSSM is lower, the prior regularization (KL) promotes
  more canonical representations. If similar, architecture alone doesn't help.
- *Editing:* If RSSM manifold edit produces comparable %swap obs change, the failure mode replicates.
  If substantially lower, the structured prior or separate h/s makes the manifold harder to navigate.
- *Generative sensitivity:* Probe ≈ random ≪ PCA for GRU — expect similar for RSSM unless stochastic
  `s` component is better aligned with the decoder.

See `research/scratch/` for the write-up and `research/findings/` for promoted conclusions.

**RSSM-specific observation from h/s decomposition:**
Compare position R² from `lin det` vs `lin stoch` — determines whether position information
concentrates in the deterministic or stochastic component. Theory predicts `s` captures the
world state beyond history (the "compact" information), but in practice the GRU-cell in RSSM
may push most position info into `h`.